# 02 — Bronze → Silver

## Student Demo

Goal:

```text
Bronze
  ↓
Clean
  ↓
Cast Types
  ↓
Remove Duplicates
  ↓
Silver Delta
```


In [ ]:
from pyspark.sql import functions as F

CATALOG = "retail_catalog2"

customers = spark.table(f"{CATALOG}.bronze.customers")
orders = spark.table(f"{CATALOG}.bronze.orders")
products = spark.table(f"{CATALOG}.bronze.products")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")


## 1. Clean Customers


In [ ]:
customers_silver = (
    customers
    .dropDuplicates(["customer_id"])
    .withColumn("email", F.lower(F.trim("email")))
    .withColumn("city", F.initcap(F.trim("city")))
    .filter(F.col("customer_id").isNotNull())
)

display(customers_silver)


## 2. Clean Orders


In [ ]:
orders_silver = (
    orders
    .dropDuplicates(["order_id"])
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .withColumn(
        "sales_amount",
        F.col("quantity") * F.col("unit_price")
    )
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("quantity") > 0)
)

display(orders_silver)


## 3. Clean Products


In [ ]:
products_silver = (
    products
    .dropDuplicates(["product_id"])
    .withColumn("product_name", F.trim("product_name"))
    .withColumn("category", F.initcap(F.trim("category")))
    .withColumn("unit_price", F.col("unit_price").cast("double"))
    .filter(F.col("product_id").isNotNull())
)

display(products_silver)


## 4. Write Silver


In [ ]:
(
    customers_silver.write
    .format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.silver.customers")
)

(
    orders_silver.write
    .format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.silver.orders")
)

(
    products_silver.write
    .format("delta").mode("overwrite")
    .saveAsTable(f"{CATALOG}.silver.products")
)

print("Silver tables created")


## Student explanation

> Silver is the trusted layer. Here we clean strings, cast data types, remove duplicates, reject invalid records and calculate simple derived columns.
